## I. Imports

### I.1. Import Modules

In [448]:
import numpy as np
import csv
import json

### I.1. Read CSVs  

In [449]:
old_data = []
with open('../dataset/wfp_food_prices_phl.csv', 'r') as f:
    reader = csv.reader(f, delimiter=',', quotechar='"')
    for row in reader:
        if row[0] != "date":
            old_data.append({
                'date' : row[0],
                'admin1' : row[1],
                'admin2' : row[2],
                'category' : row[7],
                'commodity' : row[8],
                'unit' : row[10],
                'pricetype' : row[12],
                'price' : float(row[14]),
            })


## II. Preprocessing

### II.1. Date Processing

In [450]:
data = []
for i in old_data:
    (year, month, day) = i['date'].split('-')
    if int(year) >= 2020:
        i['year'] = int(year)
        i['month'] = int(month)
        data.append(i)
        

### II.2. Region Processing

In [451]:
region_map = {
    'National Capital region' : 0,
    'Region I' : 1,
    'Region II' : 2,
    'Region III' : 3,
    'Region IV-A' : 4,
    'Region IV-B' : 5,
    'Region V' : 6,
    'Region VI' : 7,
    'Region VII' : 8,
    'Region VIII' : 9,
    'Region IX' : 10,
    'Region X' : 11,
    'Region XI' : 12,
    'Region XII' : 13,
    'Region XIII' : 14,
    'Cordillera Administrative region' : 15,
    'Autonomous region in Muslim Mindanao' : 16,
}

for x in data:
    x['region'] = region_map[x['admin1']]

### II.3. Province ID

In [452]:
province_list = []

for x in data:
    if x['admin2'] not in province_list:
        province_list.append(x['admin2'])

for x in data:
    x['province_id'] = province_list.index(x['admin2'])

### II.4. Data Binning

Split the data between Rice and Non-Rice Products.

In [453]:
rice_data = []
non_rice_data = []
provinces = ['Metropolitan Manila', 'Nueva Ecija', 'Iloilo', 'South Cotabato', 'Basilan', 'Maguindanao', 'Ifugao', 'Kalinga', 'Ilocos Norte', 'Ilocos Sur', 'Cagayan', 'Isabela',
 'Quirino', 'Bataan', 'Bulacan', 'Batangas', 'Cavite', 'Laguna', 'Quezon', 'Rizal', 'Marinduque', 'Occidental Mindoro', 'Oriental Mindoro', 'Palawan', 'Romblon', 'Zamboanga del Sur',
 'Albay', 'Catanduanes', 'Masbate', 'Sorsogon', 'Aklan', 'Guimaras', 'Negros Occidental', 'Bohol', 'Siquijor', 'Biliran', 'Leyte', 'Northern Samar', 'Samar', 'Southern Leyte', 'Bukidnon',
 'Camiguin', 'Lanao del Norte', 'Misamis Occidental', 'Misamis Oriental', 'Davao Oriental', 'Davao del Norte', 'North Cotabato', 'Sarangani', 'Agusan del Norte', 'Surigao del Norte',
 'Surigao del Sur']

for x in data:
    if x['commodity'] == 'Rice (regular, milled)' and x['pricetype'] == 'Retail':
        rice_data.append(x)
    else:
        non_rice_data.append(x)

### II.5. Monthly Average Price of Rice (National)
Contains calculations of:
- Monthly Average Price of Rice (National)
- Last Month's % Change of Price of Rice

In [454]:
monthly_prices = dict()

# y -> m -> [sum, n]
for y in range(2020, 2027):
    for m in range(1, 13):
        if y not in monthly_prices.keys():
            monthly_prices[y] = dict()
        monthly_prices[y][m] = [0, 0]

for x in rice_data:
    monthly_prices[x['year']][x['month']][0] += float(x['price'])
    monthly_prices[x['year']][x['month']][1] += 1

for y in range(2020, 2027):
    for m in range(1, 13):
        if monthly_prices[y][m][1] != 0:
            monthly_prices[y][m] = monthly_prices[y][m][0]/monthly_prices[y][m][1]

# 2020 3rd month and 4th month has no data, manual imputation
monthly_prices[2020][3] = monthly_prices[2020][2] + (monthly_prices[2020][5] - monthly_prices[2020][2])/3
monthly_prices[2020][4] = monthly_prices[2020][2] + 2*(monthly_prices[2020][5] - monthly_prices[2020][2])/3

# with open('dump/out1.txt', 'w') as f:
#     json.dump(monthly_prices, f, indent='\t')

for x in rice_data:
    x['national_average'] = monthly_prices[x['year']][x['month']]

    # Last Month's Percent Change of Price of Rice
    # feb = jan-dec/dec
    if x['month'] == 2:
        if x['year'] -1 != 2019:
            x['nat_avg_change_percent'] = (monthly_prices[x['year']][x['month']-1] - monthly_prices[x['year']-1][12])/abs(monthly_prices[x['year']-1][12])
        else:
            x['nat_avg_change_percent'] = 0
    # jan = dec-nov/nov
    elif x['month'] == 1:
        if x['year'] -1 != 2019:
            x['nat_avg_change_percent'] = (monthly_prices[x['year']-1][12] - monthly_prices[x['year']-1][11])/abs(monthly_prices[x['year']-1][11])
        else:
            x['nat_avg_change_percent'] = 0
    else:                   
        x['nat_avg_change_percent'] = (monthly_prices[x['year']][x['month']-1] - monthly_prices[x['year']][x['month']-2])/abs(monthly_prices[x['year']][x['month']-2])

### II.6. Province Binning

In [455]:
complete_data = []
for x in rice_data:
    if x['admin2'] in provinces:
        complete_data.append(x)

### II.7. Last Month's Provincial Price Gap 

Compared to National Average

In [456]:
province_prices = dict.fromkeys(provinces, 0)
for x in province_prices.keys():
    province_prices[x] = dict.fromkeys(range(2020, 2027), 0)
    for y in province_prices[x]:
        province_prices[x][y] = dict.fromkeys(range(1, 13), 0)

for x in complete_data:
    province_prices[x['admin2']][x['year']][x['month']] = x['price']

for x in province_prices.keys():
    province_prices[x][2020][3] = province_prices[x][2020][2] + (province_prices[x][2020][5] - province_prices[x][2020][2])/3
    province_prices[x][2020][4] = province_prices[x][2020][2] + 2*(province_prices[x][2020][5] - province_prices[x][2020][2])/3

for x in complete_data:
    # copied from II.5.
    if x['month'] == 1:
        if x['year']-1 == 2019:
            x['last_month_price_gap'] = 0
        else:
            x['last_month_price_gap'] = province_prices[x['admin2']][x['year']-1][12] - monthly_prices[x['year']-1][12]
    else:
        x['last_month_price_gap'] = province_prices[x['admin2']][x['year']][x['month']-1] - monthly_prices[x['year']][x['month']-1]


# with open('dump/out2.txt', 'w') as f:
#     json.dump(province_prices, f, indent='\t')


### II.8. L-month Backtrack

In [457]:
L = 3

l_month_back = dict.fromkeys(range(2019, 2027), 0)
for x in l_month_back.keys():
    l_month_back[x] = [0]*12

for x in range(2020, 2027):
    for m in range(12):
        l_month_back[x][m] = monthly_prices[x][m+1]

print(l_month_back)

for x in complete_data:
    for y in range(3):
        x[f'{y+1}_month_back'] = l_month_back[x['year'] - (1 if x['month'] - y - 1 <= 0 else 0)][(x['month'] - y - 1) % 12]

{2019: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 2020: [37.425000000000004, 36.408, 37.00214814814815, 37.596296296296295, 38.190444444444445, 38.24919540229886, 38.198863636363654, 37.88611111111111, 37.60581395348838, 36.866477272727266, 36.4595294117647, 36.258539325842705], 2021: [37.2540425531915, 37.229347826086965, 37.36720000000002, 37.48220779220781, 37.614556962025326, 37.582207792207804, 37.724868421052655, 37.821973684210526, 37.98473684210527, 37.78605263157896, 37.58217948717949, 37.62961038961039], 2022: [38.4040243902439, 38.666, 38.640641025641024, 38.75341772151898, 38.86839506172839, 38.97759493670887, 39.21937500000001, 39.42620253164558, 39.54567901234568, 39.50860759493671, 39.53481012658228, 39.612804878048784], 2023: [39.65703703703706, 39.69333333333336, 39.890499999999996, 40.34864197530866, 40.58841463414634, 40.94444444444446, 41.279375000000016, 43.008875, 47.53625000000001, 45.35825000000001, 46.63375, 48.213999999999984], 2024: [49.58395061728396, 50.44812500

## III. Output


### Temporary Output

In [458]:
# with open('dump/out.txt', 'w') as f:
#     json.dump(complete_data, f, indent='\t')

### Output as CSV

In [459]:
with open('../dataset/processed_train.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for x in complete_data:
        if x['year'] < 2025 and not (x['year'] == 2020 and x['month'] < 5):
            row = [
                x['month'],
                x['nat_avg_change_percent'],
                x['last_month_price_gap'],
            ]

            for l in range(L):
                row.append(x[f'{l+1}_month_back'])

            row.append(x['price'])

            writer.writerow(row)

with open('../dataset/processed_tune.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for x in complete_data:
        if x['year'] == 2025:
            row = [
                x['month'],
                x['nat_avg_change_percent'],
                x['last_month_price_gap'],
            ]

            for l in range(L):
                row.append(x[f'{l+1}_month_back'])

            row.append(x['price'])
            writer.writerow(row)

with open('../dataset/processed_test.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for x in complete_data:
        if x['year'] == 2026:
            row = [
                x['month'],
                x['nat_avg_change_percent'],
                x['last_month_price_gap'],
            ]

            for l in range(L):
                row.append(x[f'{l+1}_month_back'])

            row.append(x['price'])
            writer.writerow(row)